# Inference in Linear Dynamical Systems

In a [previous post](https://pschulam.github.io/posts/the_kalman_filter/), I described the basic ideas behind the Kalman filter for tracking in linear dynamical systems and implemented the algorithm in Python. Although the Kalman filter is useful for tracking, it turns out to be a key step in several other analyses that we may want to perform for time series data. In this post, I'll show how an understanding of the Kalman filter lays foundations for three operations: `simulate`, `score`, and `smooth`.

<!-- TEASER_END -->

# Kalman Filter Recap

The Kalman filter is an algorithm for *tracking* the latent state of a system over time given a sequence of measurements. The filter recursively applies two fundamental operations (`correct` and `predict`), and allows us to compute

$$
\alpha_k(x) = p(x_k = x \mid y_{1:k}),
$$

where $y_{1:k}$ is a sequence of measurements observed at times $1, \ldots, k$. I'll refer to these distributions as the *filtered distributions* for the latent state. If this notation is unfamiliar to you, you can read my [previous post](https://pschulam.github.io/posts/the_kalman_filter/) to get up to speed. In the rest of this post, we'll see what we can actually do with these filtered distributions.

As a starting point, here is the Python code I developed in the previous post.

In [1]:
from functools import partial

import numpy as np

from numpy.linalg import cholesky
from scipy.linalg import solve_triangular

solve_tril = partial(solve_triangular, lower=True)
solve_triu = partial(solve_triangular, lower=False)

'''
Assume that the system variable has the following structure.

system = {
    'dyn': (F, G, Q),
    'obs': (H, R)
}
'''

def kalman_filter(system, m, C, Y, U):
    for y, u in zip(Y, U):
        m, C = kalman_step(system, m, C, y, u)
    return m, C

def kalman_step(system, m, C, y, u):
    m_c, C_c = correct(*system['obs'], m, C, y)
    m_p, C_p = predict(*system['dyn'], m_c, C_c, u)
    return m_p, C_p

def correct(H, R, m, C, y):
    L = cholesky(H @ C @ H.T + R)
    Z = solve_tril(L, H @ C)
    X = solve_triu(L.T, Z)
    m_c = m + X @ (y - H @ m)
    C_c = C - Z @ Z.T
    return m_c, C_c

def predict(F, G, Q, m, C, u):
    m_p = F @ m + G @ u
    C_p = F @ C @ F.T + Q
    return m_p, C_p

## Simulate

We'll start with the simplest operation: `simulate`. This doesn't actually build on the Kalman filter, but it does connect with it in an important way, which we'll get to soon. This operation is conceptually straightforward: we simply want to simulate latent states and measurements from a linear dynamical system. In addition to the system, this operation requires an initial belief over the first latent state and a sequence of inputs. We'll assume that the initial belief is Gaussian with parameters $m$ and $C$ and that the sequence of inputs $u_{1:T}$ is the same length as the number of steps we'd like to simulate.

To implement the operation, I'm going to break `simulate` down into two simpler operations: `measure` and `advance`. `measure` takes a latent state and produces a noisy observation of it according to the system's measurement model. `advance` takes a latent state and produces the conditional distribution over the subsequent state according to the system's dynamics model.

Here is some Python code implementing `simulate`.

In [2]:
mvn = np.random.multivariate_normal

def simulate(system, m, C, U):
    n_steps = len(U)
    X = np.zeros((n_steps, n_hidden(system)))
    Y = np.zeros((n_steps, n_observed(system)))
    for i, u in enumerate(U):
        X[i] = mvn(m, C)
        Y[i] = measure(*system['obs'], x)
        m, C = advance(*system['dyn'], x, u)
    return X, Y

def measure(H, R, x):
    return mvn(H @ x, R)

def advance(F, G, Q, x, u):
    return F @ x + G @ u, C

def n_hidden(system):
    F, _, _ = system['dyn']
    return len(F)

def n_inputs(system):
    _, G, _ = system['dyn']
    return G.shape[1]

def n_observed(system):
    H, _ = system['obs']
    return len(H)